In [2]:
import sqlite3

with sqlite3.connect(r"D:\Data Analytics\healthcare-rcm-charge-reimbursement-analytics\healthcare_rcm.db") as conn:
    cursor = conn.cursor()


In [3]:
import pandas as pd

data_types = {
    "Rndrng_Prvdr_CCN": "str",
    "Rndrng_Prvdr_State_FIPS": "str",
    "Rndrng_Prvdr_Zip5": "str",
    "DRG_Cd": "str"
}
df = pd.read_csv('../data/MUP_INP_RY26_P03_V10_DY24_PrvSvc.csv', dtype=data_types)
df["Total_Charges"] = df["Tot_Dschrgs"] * df["Avg_Submtd_Cvrd_Chrg"]
df["Total_Mdcr_Pymt"] = df["Tot_Dschrgs"] * df["Avg_Mdcr_Pymt_Amt"]
df["Chrg_to_Mdcr_Pymt_Ratio"] = df["Total_Charges"] / df["Total_Mdcr_Pymt"]

In [4]:
hospitals = df[["Rndrng_Prvdr_CCN", "Rndrng_Prvdr_Org_Name", "Rndrng_Prvdr_City", "Rndrng_Prvdr_State_Abrvtn", "Rndrng_Prvdr_Zip5", "Rndrng_Prvdr_RUCA", "Rndrng_Prvdr_RUCA_Desc"]]
hospitals = hospitals.drop_duplicates()
print(hospitals.shape)
print(hospitals["Rndrng_Prvdr_CCN"].nunique())

(2906, 7)
2906


In [5]:
drgs = df[["DRG_Cd", "DRG_Desc"]]
drgs = drgs.drop_duplicates()
print(drgs.shape)
print(drgs["DRG_Cd"].nunique())

(540, 2)
540


In [19]:
fact_discharges = df[["Rndrng_Prvdr_CCN", "DRG_Cd", "Tot_Dschrgs", "Avg_Submtd_Cvrd_Chrg", "Avg_Tot_Pymt_Amt", "Avg_Mdcr_Pymt_Amt", "Total_Charges", "Total_Mdcr_Pymt", "Chrg_to_Mdcr_Pymt_Ratio"]]
print(fact_discharges.shape)


(145879, 9)


In [10]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS hospitals (
        Rndrng_Prvdr_CCN TEXT PRIMARY KEY,
        Rndrng_Prvdr_Org_Name TEXT,
        Rndrng_Prvdr_City TEXT,
        Rndrng_Prvdr_State_Abrvtn TEXT,
        Rndrng_Prvdr_Zip5 TEXT,
        Rndrng_Prvdr_RUCA TEXT,
        Rndrng_Prvdr_RUCA_Desc TEXT
    );
""")

hospitals.to_sql(
    name="hospitals",
    con=conn,
    if_exists="append",
    index=False
)
conn.commit()

In [13]:
cursor.execute("SELECT COUNT(*) FROM hospitals;")
print(cursor.fetchone())

(2906,)


In [14]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS drgs (
        DRG_Cd TEXT PRIMARY KEY,
        DRG_Desc TEXT
        );
""")
drgs.to_sql(
    name="drgs",
    con=conn,
    if_exists="append",
    index=False
)
conn.commit()

In [15]:
cursor.execute("SELECT COUNT(*) FROM drgs;")
print(cursor.fetchone())

(540,)


In [20]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS fact_discharges (
        Rndrng_Prvdr_CCN TEXT,
        DRG_Cd TEXT,
        Tot_Dschrgs INTEGER,
        Avg_Submtd_Cvrd_Chrg REAL,
        Avg_Tot_Pymt_Amt REAL,
        Avg_Mdcr_Pymt_Amt REAL,
        Total_Charges REAL,
        Total_Mdcr_Pymt REAL,
        Chrg_to_Mdcr_Pymt_Ratio REAL,
        PRIMARY KEY (Rndrng_Prvdr_CCN, DRG_Cd)
    )
""")
fact_discharges.to_sql(
    name="fact_discharges",
    con=conn,
    if_exists="append",
    index=False
)
conn.commit()

In [21]:
cursor.execute("SELECT COUNT(*) FROM fact_discharges;")
print(cursor.fetchone())

(145879,)


In [23]:
cursor.execute("""
    CREATE INDEX IF NOT EXISTS idx_fact_discharges_provider
    ON fact_discharges (Rndrng_Prvdr_CCN)
""")
conn.commit()

In [25]:
cursor.execute("""
    CREATE INDEX IF NOT EXISTS idx_fact_discharges_drg
    ON fact_discharges (DRG_Cd)
""")
conn.commit()

In [26]:
cursor.execute("PRAGMA index_list('fact_discharges');")
print(cursor.fetchall())

[(0, 'idx_fact_discharges_drg', 0, 'c', 0), (1, 'idx_fact_discharges_provider', 0, 'c', 0), (2, 'sqlite_autoindex_fact_discharges_1', 1, 'pk', 0)]
